# 03 — Real Input Validation

Phase 2 assumed that data was already close to its analytical types.

Phase 3 asks a more realistic question:

> What happens when a CSV contains text numbers, missing values, invalid dates, unexpected columns, and several errors at once?

We will build and inspect:

- per-field coercion,
- null policy,
- date coercion,
- `lazy=True`,
- `failure_cases`,
- `strict="filter"`,
- detailed and summary CSV reports.

## 1. Project setup

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

WindowsPath('C:/Users/Victus 16/PycharmProjects/pandera-data-quality-lab')

In [2]:
import pandas as pd
import pandera.pandas as pa

from pandera_lab import (
    load_orders_csv,
    summarize_failure_cases,
    validate_orders,
    write_failure_report,
    write_failure_summary,
)
from pandera_lab.schemas import OrderSchema

## 2. Inspect the evolved Phase-3 schema

Notice the three major changes:

1. selected fields now use `coerce=True`,
2. nullability is explicit,
3. extra columns use `strict="filter"`.

In [3]:
OrderSchema.to_schema()

<Schema DataFrameSchema(columns={'order_id': <Schema Column(name=order_id, type=DataType(int64))>, 'customer_id': <Schema Column(name=customer_id, type=DataType(str))>, 'product_id': <Schema Column(name=product_id, type=DataType(str))>, 'quantity': <Schema Column(name=quantity, type=DataType(int64))>, 'unit_price': <Schema Column(name=unit_price, type=DataType(float64))>, 'discount': <Schema Column(name=discount, type=DataType(float64))>, 'total': <Schema Column(name=total, type=DataType(float64))>, 'status': <Schema Column(name=status, type=DataType(str))>, 'order_date': <Schema Column(name=order_date, type=DataType(datetime64[ns]))>}, checks=[], parsers=[], index=None, dtype=None, coerce=False, strict=filter, name=OrderSchema, ordered=False, unique=None, report_duplicates=all, unique_column_names=False, add_missing_columns=False, title=None, description=Validated analytical shape for an order record., metadata=None, drop_invalid_rows=False)>

## 3. Load the real messy CSV without cleaning it

The ingestion layer deliberately does not parse dates or repair values.

In [4]:
raw_path = ROOT / "data" / "raw" / "orders.csv"
raw_df = load_orders_csv(raw_path)

raw_df

,order_id,customer_id,product_id,quantity,unit_price,discount,total,status,order_date,internal_note
0,1001,C001,P001,2,1200.0,0.10,2160.0,paid,2026-08-01,priority customer
1,1002,C002,P002,1,25.0,0.00,25.0,shipped,2026-08-02,NaN
2,1003,C003,P003,0,80.0,0.00,0.0,paid,2026-08-03,zero quantity
3,1004,C004,P004,3,-50.0,0.10,-135.0,pending,2026-08-04,negative price
4,1005,C005,P005,1,300.0,1.20,-60.0,paid,2026-08-05,discount over 100 percent
5,1006,C006,P006,2,75.0,0.10,100.0,shipped,2026-08-06,wrong total
6,1007,C007,P007,2,40.0,0.00,80.0,UNKNOWN,2026-08-07,invalid status
7,1008,C008,P008,1,50.0,0.00,50.0,cancelled,2026-08-08,valid row
8,1008,C009,P009,1,70.0,0.00,70.0,paid,2026-08-09,duplicate order id
9,1010,C010,P010,NaN,90.0,0.10,81.0,paid,2026-08-10,missing quantity


In [5]:
raw_df.dtypes

order_id           int64
customer_id       object
product_id        object
quantity          object
unit_price       float64
discount         float64
total            float64
status            object
order_date        object
internal_note     object
dtype: object

### Think before validating

Find examples of:

- values that are convertible,
- values that are not convertible,
- missing values,
- value-rule violations,
- invalid categories,
- invalid dates,
- extra columns.

Remember: a single row can contain more than one kind of problem.

## 4. Fail-fast validation

This is useful when you only need the first failure.

In [6]:
try:
    OrderSchema.validate(raw_df)
except pa.errors.SchemaError as exc:
    print(type(exc).__name__)
    print(str(exc).splitlines()[-1])

SchemaErrors: {
    "DATA": {
        "DATATYPE_COERCION": [
            {
                "schema": "OrderSchema",
                "column": "quantity",
                "check": "coerce_dtype('int64')",
                "error": "Error while coercing 'quantity' to type int64: Could not coerce <class 'pandas.core.series.Series'> data_container into type int64:   index failure_case0      9          NaN1     10          two"
            },
            {
                "schema": "OrderSchema",
                "column": "order_date",
                "check": "coerce_dtype('datetime64[ns]')",
                "error": "Error while coercing 'order_date' to type datetime64[ns]: Could not coerce <class 'pandas.core.series.Series'> data_container into type datetime64[ns]:   index failure_case0     11   2026-02-30"
            }
        ]
    }
}

## 5. Lazy validation

For data quality, we usually want a broader diagnosis.

`lazy=True` aggregates validation errors instead of stopping at the first one.

In [ ]:
try:
    OrderSchema.validate(raw_df, lazy=True)
except pa.errors.SchemaErrors as exc:
    lazy_error = exc
    print(type(exc).__name__)
    print("number of failure-case rows:", len(exc.failure_cases))

## 6. Inspect `failure_cases`

In [ ]:
lazy_error.failure_cases

Look for these columns in the report:

```text
schema_context
column
check
failure_case
index
```

Exact columns can vary by Pandera version/check type, but the key idea is that the error is **structured data**, not only a traceback.

## 7. Use the repository validation boundary

In [ ]:
result = validate_orders(raw_df)

print("is_valid:", result.is_valid)
print("failed_columns:", result.failed_columns)

In [ ]:
result.failure_cases

## 8. Summarize failures

In [ ]:
summary = summarize_failure_cases(result.failure_cases)
summary

This summary is more operationally useful than a giant exception string.

It can answer:

- which columns are failing,
- which checks fail most often,
- whether a source is producing repeated conversion problems.

## 9. Save detailed and summary reports

In [ ]:
detail_path = write_failure_report(
    result,
    ROOT / "reports" / "phase3_validation_errors.csv",
)

summary_path = write_failure_summary(
    result,
    ROOT / "reports" / "phase3_validation_summary.csv",
)

detail_path, summary_path

## 10. Coercion on valid raw text

Now create data that is logically valid but arrives as strings.

In [ ]:
raw_text_df = pd.DataFrame({
    "order_id": ["3001", "3002"],
    "customer_id": ["C301", "C302"],
    "product_id": ["P301", "P302"],
    "quantity": ["2", "1"],
    "unit_price": ["100.0", "25.0"],
    "discount": ["0.10", "0.00"],
    "total": ["180.0", "25.0"],
    "status": ["paid", "shipped"],
    "order_date": ["2026-08-01", "2026-08-02"],
    "internal_note": ["source metadata", "source metadata"],
})

raw_text_df.dtypes

In [ ]:
valid_result = validate_orders(raw_text_df)

print("is_valid:", valid_result.is_valid)
valid_result.data

### Compare dtypes after validation

In [ ]:
valid_result.data.dtypes

Expected conceptual transformation:

```text
order_id     "3001"       -> integer
quantity     "2"          -> integer
unit_price   "100.0"      -> float
discount     "0.10"       -> float
total        "180.0"      -> float
order_date   "2026-08-01" -> datetime
```

This is controlled coercion.

## 11. `strict="filter"`

In [ ]:
print("columns before:")
print(raw_text_df.columns.tolist())

print("\ncolumns after:")
print(valid_result.data.columns.tolist())

`internal_note` disappeared from the trusted analytical dataframe.

That is the exact Phase-3 contract:

> tolerate extra source metadata, but do not propagate it downstream.

## 12. Coercible does not mean valid

In [ ]:
negative_quantity = raw_text_df.copy()
negative_quantity.loc[0, "quantity"] = "-2"

negative_result = validate_orders(negative_quantity)
negative_result.failure_cases

`"-2"` can be converted to integer `-2`.

So type coercion can succeed while the business rule:

```text
quantity > 0
```

still fails.

## 13. Uncoercible value

In [ ]:
text_quantity = raw_text_df.copy()
text_quantity.loc[0, "quantity"] = "two"

text_result = validate_orders(text_quantity)
text_result.failure_cases

This is different from `"-2"`.

`"two"` cannot become an integer at all, so the failure is associated with coercion/type enforcement.

## 14. Invalid calendar date

In [ ]:
bad_date = raw_text_df.copy()
bad_date.loc[0, "order_date"] = "2026-02-30"

date_result = validate_orders(bad_date)
date_result.failure_cases

## 15. Missing required identifier

In [ ]:
missing_customer = raw_text_df.copy()
missing_customer.loc[0, "customer_id"] = None

null_result = validate_orders(missing_customer)
null_result.failure_cases

The contract explicitly uses `nullable=False`.

Missingness is therefore not something the pipeline silently repairs.

## 16. Important deliberate gap

Change `total` to a wildly incorrect value while keeping it numeric.

In [ ]:
wrong_total = raw_text_df.copy()
wrong_total.loc[0, "total"] = "999999"

wrong_total_result = validate_orders(wrong_total)

print("is_valid:", wrong_total_result.is_valid)
wrong_total_result.data

It still passes Phase 3.

That is intentional.

The formula:

```text
total == unit_price * quantity * (1 - discount)
```

needs a cross-column rule.

That becomes Phase 4 with dataframe-level validation.

# Phase-3 checkpoint

You should now be able to explain:

- `coerce=True`,
- why coercion is not arbitrary cleaning,
- `nullable=False`,
- fail-fast vs lazy validation,
- `SchemaError` vs `SchemaErrors`,
- `failure_cases`,
- error-report generation,
- `strict="filter"`,
- why the raw dataframe and trusted analytical dataframe are different objects.

## Next phase

Phase 4 adds custom and cross-column business rules, especially validation of the derived `total`.